# Project 90 — Local Multimodal Research Combiner
## Multiple Sources → Unified Knowledge → Research Report

**Stack:** LangChain · Ollama · ChromaDB · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb pydantic pandas

## Step 1 — Multi-Source Research Corpus

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

research_sources = {
    "paper_abstract": [
        {"title": "Attention Is All You Need",
         "text": "The dominant sequence transduction models are based on complex recurrent or "
                 "convolutional neural networks. We propose the Transformer, based solely on "
                 "attention mechanisms. Experiments show superior quality while being more "
                 "parallelizable and requiring less training time."},
        {"title": "BERT: Pre-training of Deep Bidirectional Transformers",
         "text": "We introduce BERT, designed to pre-train deep bidirectional representations "
                 "by jointly conditioning on both left and right context. BERT can be fine-tuned "
                 "with just one additional output layer for a wide range of tasks."},
    ],
    "tech_blog": [
        {"title": "Building Production RAG Systems",
         "text": "RAG combines retrieval with generation. Key components: document chunking, "
                 "embedding generation, vector storage, query processing, and answer generation. "
                 "Common pitfalls: poor chunking strategy, embedding model mismatch, "
                 "and insufficient context window utilization."},
    ],
    "meeting_notes": [
        {"title": "AI Team Sync — March 2025",
         "text": "Discussed RAG implementation timeline. Current approach uses ChromaDB with "
                 "LangChain. Performance issues with large document sets (>10K docs). "
                 "Considering hybrid retrieval with BM25. Action: benchmark by end of week."},
    ],
    "code_docs": [
        {"title": "LangChain RAG Tutorial",
         "text": "from langchain.chains import RetrievalQA; from langchain.vectorstores import "
                 "Chroma; Setup: load documents, split into chunks, embed with OllamaEmbeddings, "
                 "store in Chroma, query with RetrievalQA chain."},
    ],
}

total = sum(len(v) for v in research_sources.values())
print(f"Sources: {len(research_sources)} types, {total} documents")

## Step 2 — Index All Sources in ChromaDB

In [ ]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection("research")

doc_id = 0
for source_type, docs in research_sources.items():
    for doc in docs:
        collection.add(
            ids=[f"doc_{doc_id}"],
            documents=[doc["text"]],
            metadatas=[{"source_type": source_type, "title": doc["title"]}],
        )
        doc_id += 1

print(f"✓ Indexed {collection.count()} documents across {len(research_sources)} source types")

## Step 3 — Cross-Source Research Queries

In [ ]:
research_questions = [
    "What are the key architectures for NLP models?",
    "How should I build a production RAG system?",
    "What performance issues exist with vector databases?",
    "What code is needed to set up a RAG pipeline?",
]

class ResearchFinding(BaseModel):
    question: str
    sources_used: list[str]
    synthesis: str
    confidence: float = Field(ge=0, le=1)
    gaps: list[str] = Field(description="Information gaps that need more research")

researcher = llm.with_structured_output(ResearchFinding)

findings = []
for question in research_questions:
    results = collection.query(query_texts=[question], n_results=3)
    context = "\n\n".join(
        f"[{m['source_type']}] {m['title']}: {doc}"
        for doc, m in zip(results["documents"][0], results["metadatas"][0])
    )
    finding = researcher.invoke(
        f"Research question: {question}\n\nSources:\n{context}"
    )
    findings.append(finding)
    print(f"\nQ: {question}")
    print(f"  Sources: {finding.sources_used}")
    print(f"  Confidence: {finding.confidence:.0%}")
    print(f"  Gaps: {finding.gaps[:2]}")

## Step 4 — Generate Research Report

In [ ]:
report_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a structured research report based on these findings. "
     "Include: Executive Summary, Key Findings, Methodology, Gaps & Next Steps. "
     "Use Markdown formatting."),
    ("human", "Research findings:\n{findings}\n\nReport:")
])
report_chain = report_prompt | llm | StrOutputParser()

report = report_chain.invoke({
    "findings": json.dumps([f.model_dump() for f in findings], indent=2)
})
print("RESEARCH REPORT")
print("=" * 60)
print(report[:800])

## Step 5 — Source Coverage Matrix

In [ ]:
# Check which source types contributed to each finding
coverage = {}
for f in findings:
    for s in f.sources_used:
        coverage.setdefault(s, 0)
        coverage[s] += 1

print("SOURCE UTILIZATION")
print("=" * 40)
for source, count in sorted(coverage.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {source:<20} {bar} ({count})")

print(f"\nResearch gaps to fill:")
all_gaps = [g for f in findings for g in f.gaps]
for i, gap in enumerate(all_gaps[:5], 1):
    print(f"  {i}. {gap}")

## What You Learned
- **Multi-source research aggregation** in ChromaDB
- **Cross-reference synthesis** from diverse document types
- **Research report generation** with structured findings
- **Coverage analysis** and gap identification